# QuizForge – Demo notebook (no UI)
Runs the same pipeline as the Streamlit app: **read book → detect chapters & topics → choose scope → generate question types → sort by topic → rate & sentiment analysis**.

Run it from the `notebooks/` folder (or in Colab after uploading the whole project folder).

In [ ]:
!pip install -q -r ../requirements.txt

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os
sys.path.append(os.path.abspath(".."))   # so `import qgen` works from notebooks/

from collections import Counter
from qgen.extractor import extract_text, build_chunks, outline
from qgen.generator import QuestionGenerator
from qgen.pipeline import generate_quiz
from qgen.formatting import item_to_text
from qgen.schema import QUESTION_TYPES

## 1. Read the book and detect chapters / topics
Whole textbooks are fine; progress prints per page.

In [ ]:
BOOK_PATH = "my_textbook.pdf"   # <-- change this (PDF or DOCX, any size)

with open(BOOK_PATH, "rb") as f:
    raw_text = extract_text(BOOK_PATH, f.read())   # keeps line breaks (needed to find headings)
chunks = build_chunks(raw_text)
book = outline(chunks)
print(len(chunks), "passages")
for chapter, topics in book.items():
    print(repr(chapter or "(no chapters)"), "->", len(topics), "topics:", list(topics)[:5])

## 2. Load the Seq2Seq Transformer LLM
Only needed for *Subjective* and *MCQ* questions. The first run downloads the weights.

In [ ]:
generator = QuestionGenerator("google/flan-t5-base")
print("Running on:", generator.device)

## 3. Generate
* `QTYPES` – any of: `subjective`, `mcq`, `fill_blank`, `true_false`, `match`
* `DIFFICULTY` – `low` / `mid` / `high` · `NUM_QUESTIONS` – multiple of 5 (max 30)
* `SCOPE` – `None` = whole book, or a set of topic keys such as `{"Chapter 2: Operators › 2.1 Selection methods"}`

In [ ]:
QTYPES = ["subjective", "mcq", "fill_blank", "true_false", "match"]
DIFFICULTY, NUM_QUESTIONS, SCOPE = "mid", 15, None

items = generate_quiz(chunks, generator, DIFFICULTY, NUM_QUESTIONS, QTYPES, SCOPE,
                      progress_callback=lambda p: print(f"\rprogress: {p:.0%}", end=""))
print()
print(Counter(QUESTION_TYPES[i.qtype] for i in items))

## 4. Questions sorted by topic (with answers)

In [ ]:
from itertools import groupby
for topic, group in groupby(sorted(items, key=lambda i: (i.chapter, i.topic)), key=lambda i: (i.chapter, i.topic)):
    print(f"\n=== {' › '.join(x for x in topic if x)} ===")
    for it in group:
        print(item_to_text(it, it.uid + 1, with_answers=True))

## 5. (Optional) Feedback with sentiment analysis
Rate relevance 1–5 and add comments; a DistilBERT (SST-2) model classifies the comments.

In [ ]:
from qgen.feedback import FeedbackEntry, SentimentAnalyzer, run_sentiment, summarize_feedback

entries = [FeedbackEntry(i.uid, i.question, i.qtype, i.topic) for i in items]
entries[0].stars, entries[0].comment = 5, "Clear and exactly on topic"
entries[1].stars, entries[1].comment = 2, "Confusing and not related to the chapter"

run_sentiment(entries, SentimentAnalyzer())        # downloads ~270 MB on first use
summary = summarize_feedback(entries)
print("average relevance:", summary.avg_stars, "| sentiments:", summary.sentiment_counts)
print("flagged:", summary.flagged)
print("by topic (weakest first):", summary.by_topic)